In [1]:
!pip install --upgrade git+https://github.com/huggingface/transformers accelerate

  Cloning https://github.com/huggingface/transformers to /tmp/pip-req-build-3icq12rg
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers /tmp/pip-req-build-3icq12rg
  Resolved https://github.com/huggingface/transformers to commit ad697ec123f5133e5aae45c97c23d90ea52a1bd8
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import os
import tempfile
from pathlib import Path

import pandas as pd
from google.colab import drive


drive.mount('/content/drive')

df = pd.read_parquet("/content/drive/MyDrive/data.parquet")
print(df.shape)
df.head(1)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
(1230, 3)


,audio,title,artist
0,{'bytes': b'ID3\x04\x00\x00\x00\x00\x02\x03TCO...,Food,AWOL


In [3]:
import torch
from transformers import AudioFlamingo3ForConditionalGeneration, AutoProcessor

model_id = "nvidia/audio-flamingo-3-hf"

dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

processor = AutoProcessor.from_pretrained(model_id)
model = AudioFlamingo3ForConditionalGeneration.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=dtype,
)
model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/830 [00:00<?, ?it/s]

AudioFlamingo3ForConditionalGeneration(
  (model): AudioFlamingo3Model(
    (audio_tower): AudioFlamingo3Encoder(
      (conv1): Conv1d(128, 1280, kernel_size=(3,), stride=(1,), padding=(1,))
      (conv2): Conv1d(1280, 1280, kernel_size=(3,), stride=(2,), padding=(1,))
      (embed_positions): Embedding(1500, 1280)
      (layers): ModuleList(
        (0-31): 32 x AudioFlamingo3EncoderLayer(
          (self_attn): AudioFlamingo3Attention(
            (k_proj): Linear(in_features=1280, out_features=1280, bias=False)
            (v_proj): Linear(in_features=1280, out_features=1280, bias=True)
            (q_proj): Linear(in_features=1280, out_features=1280, bias=True)
            (out_proj): Linear(in_features=1280, out_features=1280, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=1280, out_features=5120, bias=True)
          (fc2): Linear(in_fe

In [4]:
def describe_row_with_af3(row, processor, model, prompt=None, max_new_tokens=180):
    if prompt is None:
        prompt = (
            "Describe this music in 2-3 sentences. "
            "Focus on genre, mood, energy, tempo, and instruments. "
            "Do not mention artist or track title."
        )

    audio_bytes = row["audio"]["bytes"]
    suffix = Path(row["audio"].get("path", "track.mp3")).suffix or ".mp3"

    with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as tmp:
        tmp.write(audio_bytes)
        tmp_path = tmp.name

    try:
        conversation = [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {"type": "audio", "path": tmp_path},
                ],
            }
        ]

        inputs = processor.apply_chat_template(
            conversation,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
        ).to(model.device)

        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

        decoded = processor.batch_decode(
            outputs[:, inputs.input_ids.shape[1]:],
            skip_special_tokens=True
        )

        return decoded[0] if len(decoded) > 0 else ""

    finally:
        if os.path.exists(tmp_path):
            os.remove(tmp_path)

In [6]:
row = df.iloc[0]
description = describe_row_with_af3(row, processor, model)
print(description)

RuntimeError: Input type (float) and bias type (c10::BFloat16) should be the same